# Stable Diffusion WebUI — Full Colab Edition

Полная сборка AUTOMATIC1111 для Google Colab. Запустите ячейки сверху вниз.

In [ ]:
#@title 1. Проверка среды и GPU
import os, sys, shutil, subprocess, platform
from IPython.display import clear_output

print('Python:', sys.version)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu if gpu else 'GPU НЕ ОБНАРУЖЕН')
if not gpu:
    raise RuntimeError('Выберите среду выполнения с GPU: Среда выполнения → Сменить среду выполнения → T4/L4/A100')

In [ ]:
#@title 2. Установка WebUI и системных инструментов
#@markdown Репозиторий скачивается заново только при отсутствии. Модели и настройки можно хранить на Google Drive.
USE_DRIVE = True #@param {type:'boolean'}
MOUNT_PATH = '/content/drive' #@param {type:'string'}
ROOT = '/content/drive/MyDrive/SD-WebUI' if USE_DRIVE else '/content'
WEBUI_DIR = f'{ROOT}/stable-diffusion-webui'

if USE_DRIVE:
    from google.colab import drive
    drive.mount(MOUNT_PATH, force_remount=False)

os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(WEBUI_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AUTOMATIC1111/stable-diffusion-webui.git', WEBUI_DIR], check=True)
else:
    subprocess.run(['git', '-C', WEBUI_DIR, 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', WEBUI_DIR, 'reset', '--hard', 'origin/master'], check=True)
    subprocess.run(['git', '-C', WEBUI_DIR, 'pull', '--ff-only'], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2', 'rsync', 'libgl1'], check=False)

# Use the versions officially pinned by the WebUI repository; do not replace PyTorch blindly.
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', '-q', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', f'{WEBUI_DIR}/requirements_versions.txt'], check=True)
print('Installed WebUI:', WEBUI_DIR)

In [ ]:
#@title 3. Расширения
#@markdown Установятся популярные и активно поддерживаемые расширения.
CONTROLNET = True #@param {type:'boolean'}
ADetailer = True #@param {type:'boolean'}
REGIONAL_PROMPTER = True #@param {type:'boolean'}
DYNAMIC_PROMPTS = True #@param {type:'boolean'}
WILDcards = True #@param {type:'boolean'}
IMAGE_BROWSER = True #@param {type:'boolean'}
TAGCOMPLETE = True #@param {type:'boolean'}

extensions = []
if CONTROLNET: extensions.append(('sd-webui-controlnet','https://github.com/Mikubill/sd-webui-controlnet.git'))
if ADetailer: extensions.append(('adetailer','https://github.com/Bing-su/adetailer.git'))
if REGIONAL_PROMPTER: extensions.append(('regional-prompter','https://github.com/hako-mikan/sd-webui-regional-prompter.git'))
if DYNAMIC_PROMPTS: extensions.append(('dynamic-prompts','https://github.com/adieyal/sd-dynamic-prompts.git'))
if WILDcards: extensions.append(('wildcards','https://github.com/AUTOMATIC1111/stable-diffusion-webui-wildcards.git'))
if IMAGE_BROWSER: extensions.append(('stable-diffusion-webui-images-browser','https://github.com/AlUlkesh/stable-diffusion-webui-images-browser.git'))
if TAGCOMPLETE: extensions.append(('a1111-sd-webui-tagcomplete','https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git'))

ext_dir = os.path.join(WEBUI_DIR, 'extensions')
os.makedirs(ext_dir, exist_ok=True)
for name, url in extensions:
    target = os.path.join(ext_dir, name)
    if os.path.exists(target):
        subprocess.run(['git', '-C', target, 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, target], check=True)
print(f'Установлено расширений: {len(extensions)}')

In [ ]:
#@title 4. Модели
#@markdown Выберите модель. Прямые ссылки могут быть Hugging Face или CivitAI; скачивание начнётся только для выбранной опции.
MODEL = 'SDXL_BASE_1.0' #@param ['None','SDXL_BASE_1.0','SD_1.5','RECOMMENDED_SDXL','CUSTOM_URL']
CUSTOM_URL = '' #@param {type:'string'}
CUSTOM_FILENAME = '' #@param {type:'string'}

models_dir = os.path.join(WEBUI_DIR, 'models', 'Stable-diffusion')
os.makedirs(models_dir, exist_ok=True)
urls = {
    'SDXL_BASE_1.0': ('https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors','sd_xl_base_1.0.safetensors'),
    'SD_1.5': ('https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors','v1-5-pruned-emaonly.safetensors'),
    'RECOMMENDED_SDXL': ('https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors','sd_xl_base_1.0.safetensors')
}
if MODEL == 'CUSTOM_URL' and CUSTOM_URL:
    url = CUSTOM_URL
    filename = CUSTOM_FILENAME or CUSTOM_URL.rstrip('/').split('/')[-1].split('?')[0]
elif MODEL in urls:
    url, filename = urls[MODEL]
else:
    url = filename = None

if url:
    target = os.path.join(models_dir, filename)
    if not os.path.exists(target):
        subprocess.run(['aria2c','-x','16','-s','16','-k','1M','-c','-d',models_dir,'-o',filename,url], check=True)
    print('Модель:', target)

In [ ]:
#@title 5. Запуск WebUI
SHARE_METHOD = 'gradio' #@param ['gradio','cloudflared','localtunnel','none']
LOWVRAM = False #@param {type:'boolean'}
MEDVRAM = True #@param {type:'boolean'}
XFORMERS = True #@param {type:'boolean'}
API = True #@param {type:'boolean'}
THEME = 'dark' #@param ['dark','light']
CUSTOM_ARGUMENTS = '' #@param {type:'string'}

args = ['python', 'launch.py', '--enable-insecure-extension-access', '--listen', '--xformers' if XFORMERS else '', '--api' if API else '', '--theme', THEME]
args = [a for a in args if a]
if LOWVRAM: args.append('--lowvram')
if MEDVRAM: args.append('--medvram')
if SHARE_METHOD == 'gradio': args.append('--share')
if SHARE_METHOD == 'cloudflared':
    subprocess.run(['bash','-lc','curl -Lo /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared'], check=True)
    subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://127.0.0.1:7860'], stdout=open('/tmp/cloudflared.log','w'), stderr=subprocess.STDOUT)
if SHARE_METHOD == 'localtunnel':
    subprocess.run(['npm','install','-g','localtunnel'], check=False)
    subprocess.Popen(['lt','--port','7860'], stdout=open('/tmp/localtunnel.log','w'), stderr=subprocess.STDOUT)
if CUSTOM_ARGUMENTS: args += CUSTOM_ARGUMENTS.split()

os.chdir(WEBUI_DIR)
subprocess.run(args, check=True)